# Correzione del bias delle immatricolazioni di flotta (ACI)

## Il problema

La domanda EV per sezione parte dal **totale provinciale** di veicoli
elettrici/ibridi plug-in (ACI 2025, `domanda_ricarica_2025_per_provincia.csv`),
poi disaggregato alle sezioni con il peso IDI×popolazione. Ma il totale ACI è
per **provincia di immatricolazione**, non di circolazione reale.

Le società di noleggio a lungo termine e leasing (NLT: Arval, Leasys, ALD…)
immatricolano in massa le proprie flotte in poche province con vantaggi
amministrativi/fiscali — storicamente **Trento, Bolzano, Aosta** e alcuni poli
come **Firenze/Livorno**. Quelle auto però circolano in tutta Italia. Il
risultato: quelle province risultano con una densità EV assurda, e la classifica
finale dei "deserti di ricarica" si concentra lì per un artefatto contabile
(il fenomeno della *«città della targa»*).

## Cosa verifica e fa questo notebook

1. **Esiste un dato ACI sulla finalità d'uso dei veicoli?** (per separare le
   flotte dalle auto dei residenti).
2. **Diagnosi** quantitativa del bias sui dati attuali del progetto.
3. **Correzione**: deflazione della quota-flotta e sua redistribuzione, con
   conservazione esatta del totale nazionale.
4. **Ri-disaggregazione** a livello di sezione, riusando i pesi IDI×popolazione
   già calcolati (nessun ricalcolo dell'IDI).

Output: `domanda_ricarica_2025_per_provincia_CORRETTA.csv` e
`domanda_ricarica_2025_per_sezione_CORRETTA.csv`.

In [1]:
import io, csv, json, glob, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DIR = Path(".")
ZIP_DIR = DIR.parent / "parco_circolante_ACI_MIT"    # 2019 ACI/MIT, con colonna 'uso'
DOMANDA_PROV = DIR / "domanda_ricarica_2025_per_provincia.csv"
GAP_PARQUET = DIR / "sezioni_gap_score.parquet"
CACHE_FLOTTE = DIR / "quota_flotte_ev_2019_per_provincia.json"

def norm(s):
    s = str(s).upper().strip()
    s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("ascii")
    s = s.replace("'"," ").replace("-"," ").replace("/"," ")
    return re.sub(r"\s+"," ", s).strip()

# nomi provincia 2019 (ACI/MIT) -> nomi provincia 2025 (ACI OPV), dopo norm()
ALIAS_2019_2025 = {
    "BARLETTA ANDRIA TRANI": "BARLETTA TRANI",
    "BOLZANO BOZEN": "BOLZANO",
    "FORLI": "FORLI CESENA",
    "MONZA E DELLA BRIANZA": "MONZA BRIANZA",
    "PESARO": "PESARO E URBINO",
    "REGGIO DI CALABRIA": "REGGIO CALABRIA",
    "REGGIO NELL EMILIA": "REGGIO EMILIA",
    "VERBANIA": "VERBANO CUSIO OSSOLA",
}
def prov_key(nome):
    n = norm(nome)
    return ALIAS_2019_2025.get(n, n)


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1. Esiste un dato ACI sulla finalità d'uso?

Tre fonti ACI usate/disponibili nel progetto:

| Fonte | Livello | Anno | Campo "uso"? |
|---|---|---|---|
| ACI *Open Parco Veicoli* (`opv.aci.it`) — stock usato per la domanda 2025 | provincia | 2025 | **no** (lo scraper estrae solo alimentazione × provincia) |
| ACI LOD *prime iscrizioni* (`aci_lod_2024.csv`) — validazione β | comune | 2024 | **no** (solo alimentazione) |
| **ACI/MIT *Parco Circolante*** (`dati.mit.gov.it`) — già scaricato in `parco_circolante_ACI_MIT/` | veicolo singolo | 2019 | **SÌ** — colonna `uso` |

Il parco circolante ACI/MIT 2019 è l'unica fonte con la **finalità d'uso** per
veicolo. I valori di `uso` includono `PROPRIO` (auto dei residenti) e
`DI TERZI DA LOCARE SENZA CONDUC.` (noleggio a lungo termine / leasing = le
flotte). È del 2019 — più vecchio dello stock 2025 — ma è l'unico dato che
misura direttamente la composizione per uso, e la geografia delle
immatricolazioni di flotta è strutturale e stabile nel tempo.

⚠️ **Attenzione al formato dei file 2019**: cinque regioni (Sicilia, Calabria,
Abruzzo, Molise, Basilicata) hanno 12 colonne, le altre 13 (una colonna-codice
iniziale in più). Le posizioni di `uso`/`provincia`/`alimentazione` vanno quindi
rilevate per file, altrimenti il Sud viene letto sulle colonne sbagliate e
sparisce dal conteggio.

In [2]:
FLEET_USI = {"DI TERZI DA LOCARE SENZA CONDUC.", "DI TERZI DA NOLEGGIO CON CONDUC."}

def calcola_quota_flotte():
    '''Legge i 20 file regionali ACI/MIT 2019 (~54M righe), filtra EV/ibridi e
    calcola per provincia: totale EV e quota immatricolata come flotta.
    Rileva per file lo scostamento di colonna (12 vs 13 colonne).'''
    agg = {}
    for z in sorted(glob.glob(str(ZIP_DIR / "*.csv.zip"))):
        import zipfile
        with zipfile.ZipFile(z) as zf:
            with zf.open(zf.namelist()[0]) as f:
                for line in io.TextIOWrapper(f, encoding="latin1"):
                    if "ELETT" not in line and "IBRID" not in line:
                        continue
                    row = next(csv.reader([line]))
                    off = 1 if row and row[0].strip().isdigit() else 0  # 13 vs 12 colonne
                    if len(row) < 7 + off:
                        continue
                    uso, prov, alim = row[2+off], row[3+off], row[6+off]
                    if not (alim == "ELETTR" or alim.startswith("IBRIDO")):
                        continue
                    a = agg.setdefault(prov, [0, 0])
                    a[0] += 1
                    if uso in FLEET_USI:
                        a[1] += 1
    return {p: {"ev_2019": t, "ev_flotta_2019": fl, "quota_flotta_2019": fl/t}
            for p, (t, fl) in agg.items()}

if CACHE_FLOTTE.exists():
    quote = json.load(open(CACHE_FLOTTE, encoding="utf-8"))
    print(f"Quote-flotta caricate dalla cache ({len(quote)} province).")
else:
    print("Calcolo quote-flotta dai file 2019 (~1-2 min)...")
    quote = calcola_quota_flotte()
    json.dump(quote, open(CACHE_FLOTTE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

tot_ev19 = sum(v["ev_2019"] for v in quote.values())
tot_fl19 = sum(v["ev_flotta_2019"] for v in quote.values())
print(f"EV/ibridi 2019 totali: {tot_ev19:,} | quota flotta nazionale: {tot_fl19/tot_ev19:.1%}")

tab = pd.DataFrame([(p, v["ev_2019"], v["quota_flotta_2019"]) for p,v in quote.items()],
                   columns=["provincia_2019","ev_2019","quota_flotta_2019"])
tab = tab[tab.ev_2019>=200].sort_values("quota_flotta_2019", ascending=False)
print("\nProvince con più alta quota di EV immatricolati come flotta (2019):")
display(tab.head(12).reset_index(drop=True))


Quote-flotta caricate dalla cache (107 province).
EV/ibridi 2019 totali: 409,397 | quota flotta nazionale: 13.5%

Province con più alta quota di EV immatricolati come flotta (2019):


,provincia_2019,ev_2019,quota_flotta_2019
0,TRENTO,34904,0.888580
1,FIRENZE,19172,0.620227
2,BOLZANO-BOZEN,7883,0.566028
3,LIVORNO,2794,0.544023
4,REGGIO NELL'EMILIA,5062,0.152311
5,TORINO,17281,0.079567
6,FERMO,568,0.056338
7,RIMINI,2432,0.051809
8,AOSTA,1344,0.037202
9,SASSARI,1219,0.036095


Il risultato è netto: nel 2019 **quasi 9 EV/ibridi su 10 immatricolati a
Trento erano flotte** (88,9%), contro una media nazionale del ~13,5%. Firenze,
Bolzano e Livorno seguono con oltre il 50%. Sono esattamente le province che
oggi risultano gonfiate. Il Sud, al contrario, ha quote di flotta quasi nulle
(0–2%): sono immatricolazioni prevalentemente di residenti.

## 2. Diagnosi del bias sui dati attuali del progetto

Calcoliamo, dai dati **attuali** (domanda 2025 aggregata dalle sezioni), il
numero di EV stimati ogni 1.000 residenti in età di guida per provincia, e lo
confrontiamo con la quota-flotta 2019.

Attenzione a non aspettarsi una correlazione globale forte: la quota-flotta
**non** è un driver generale della densità EV. La maggior parte delle province
ad alta densità è tale per ragioni legittime (ricchezza, urbanizzazione:
Milano, Roma, Torino, Como). Il bias è un **fenomeno di coda**: distorce pochi
casi estremi portandoli a livelli implausibili. È lì che va misurato.

In [3]:
g = pq.read_table(GAP_PARQUET,
    columns=["PROVINCIA_EV_2025","popolazione_eta_guida_stimata","veicoli_da_ricaricare_stimati",
             "flag_eleggibile_EV"]).to_pandas()
g = g[g.flag_eleggibile_EV==True]
prov = (g.groupby("PROVINCIA_EV_2025")
          .agg(pop_guida=("popolazione_eta_guida_stimata","sum"),
               ev=("veicoli_da_ricaricare_stimati","sum")).reset_index())
prov["ev_per_1000"] = prov.ev / prov.pop_guida * 1000
prov["key"] = prov.PROVINCIA_EV_2025.map(prov_key)

qmap = {prov_key(p): v["quota_flotta_2019"] for p,v in quote.items()}
prov["quota_flotta_2019"] = prov.key.map(qmap)
non_match = prov[prov.quota_flotta_2019.isna()]
print(f"Province senza quota-flotta 2019 (dopo alias): {len(non_match)} -> {non_match.PROVINCIA_EV_2025.tolist()}")

from scipy.stats import spearmanr
med = prov.ev_per_1000.median()
rho = spearmanr(prov.ev_per_1000, prov.quota_flotta_2019).statistic
print(f"Correlazione (Spearman) EV/1000 vs quota-flotta, tutte le province: {rho:.3f}  (debole: e' un effetto di coda)")
print(f"Mediana nazionale EV/1000: {med:.1f}\n")
# quanto e' estremo il fenomeno: province oltre 2x la mediana e loro quota-flotta
outlier = prov[prov.ev_per_1000 > 2*med].sort_values("ev_per_1000", ascending=False)
print(f"Province con EV/1000 oltre il DOPPIO della mediana ({2*med:.0f}): {len(outlier)}")
display(prov.sort_values("ev_per_1000", ascending=False)
          .head(10)[["PROVINCIA_EV_2025","ev_per_1000","quota_flotta_2019"]]
          .round({"ev_per_1000":1,"quota_flotta_2019":3}).reset_index(drop=True))


Province senza quota-flotta 2019 (dopo alias): 0 -> []


Correlazione (Spearman) EV/1000 vs quota-flotta, tutte le province: 0.076  (debole: e' un effetto di coda)
Mediana nazionale EV/1000: 25.1

Province con EV/1000 oltre il DOPPIO della mediana (50): 7


,PROVINCIA_EV_2025,ev_per_1000,quota_flotta_2019
0,TRENTO,248.6,0.889
1,AOSTA,189.6,0.037
2,FIRENZE,120.7,0.620
3,BOLZANO,76.3,0.566
4,REGGIO EMILIA,59.1,0.152
5,TORINO,52.2,0.080
6,ROMA,51.8,0.034
7,COMO,46.1,0.006
8,MILANO,44.8,0.011
9,VARESE,44.1,0.001


Lettura corretta del risultato:

- La correlazione globale è **debole** (~0,08): la quota-flotta non spiega la
  densità EV in generale. Milano, Roma, Torino, Como sono in alto per ricchezza
  e urbanizzazione, con quota-flotta bassissima (1–8%) — sono valori **legittimi**.
- Il bias è invece nella **coda estrema**: poche province con densità EV
  implausibile *e* quota-flotta altissima. **Trento** ne è il caso limite:
  ~249 EV/1000 (10× la mediana) con l'89% di immatricolazioni di flotta, e da
  sola "spiega" il 6,6% della domanda EV nazionale pur avendo lo 0,8% della
  popolazione. Firenze (62%), Bolzano (57%) e Livorno (54%) seguono.
- **Aosta** è l'unica ad alta densità *senza* alta quota-flotta (≈4%): la sua
  inflazione ha un'origine diversa (immatricolazioni di concessionari / km0 per
  i vantaggi fiscali regionali, registrate come `PROPRIO`). La correzione basata
  sull'uso quindi **non** la tocca — resta tra i limiti.

Ci si aspetta perciò una correzione **chirurgica**: deve abbattere Trento/
Firenze/Bolzano/Livorno e lasciare quasi invariate le province legittimamente
alte.

## 3. Correzione: deflazione e redistribuzione

Per ogni provincia $p$ scomponiamo il totale EV 2025 in quota residenti e quota
flotta usando la composizione 2019:

$$EV^{privati}_p = EV_p \cdot (1 - f_p), \qquad EV^{flotta}_p = EV_p \cdot f_p$$

dove $f_p$ è la quota-flotta 2019. Le auto private restano ancorate alla loro
provincia (sono residenti). Le auto di flotta di tutte le province vengono messe
in un **unico bacino nazionale** e ridistribuite dove le auto effettivamente
circolano. Come chiave di redistribuzione usiamo la **popolazione in età di
guida** (scelta neutra: «le auto circolano dove stanno le persone»):

$$EV^{corretto}_p = EV^{privati}_p + \Big(\textstyle\sum_q EV^{flotta}_q\Big)\cdot
\frac{pop\_guida_p}{\sum_q pop\_guida_q}$$

Per costruzione il **totale nazionale è conservato esattamente**. Le poche
province senza $f_p$ affidabile (2019 con meno di ~200 EV) usano la media
nazionale come ripiego prudente.

In [4]:
naz = tot_fl19 / tot_ev19
ev19_map = {prov_key(p): v["ev_2019"] for p,v in quote.items()}
prov["ev_2019"] = prov.key.map(ev19_map)
# usa la quota provinciale se stimata su >=200 EV nel 2019, altrimenti media nazionale
prov["f"] = np.where((prov.ev_2019 >= 200) & prov.quota_flotta_2019.notna(),
                     prov.quota_flotta_2019, naz)

prov["ev_privati"] = prov.ev * (1 - prov.f)
prov["ev_flotta"]  = prov.ev * prov.f
pool = prov.ev_flotta.sum()
prov["ev_flotta_redistr"] = pool * prov.pop_guida / prov.pop_guida.sum()
prov["ev_corretto"] = prov.ev_privati + prov.ev_flotta_redistr
prov["ev_per_1000_corr"] = prov.ev_corretto / prov.pop_guida * 1000

# variante di sensibilità: redistribuzione proporzionale agli EV privati locali
prov["ev_flotta_redistr_alt"] = pool * prov.ev_privati / prov.ev_privati.sum()
prov["ev_corretto_alt"] = prov.ev_privati + prov.ev_flotta_redistr_alt

assert abs(prov.ev_corretto.sum() - prov.ev.sum()) < 1.0, "totale nazionale non conservato"
print(f"Totale nazionale: originale {prov.ev.sum():,.0f} -> corretto {prov.ev_corretto.sum():,.0f} (conservato)")
print(f"Bacino flotte redistribuito: {pool:,.0f} EV ({pool/prov.ev.sum():.1%} del totale)\n")

conf = prov.sort_values("ev_per_1000", ascending=False).head(10).copy()
print("Effetto sulle province più gonfiate (EV/1000 e totale EV: prima -> dopo):")
for _,r in conf.iterrows():
    print(f"  {r.PROVINCIA_EV_2025:<22} f={r.f:5.1%}  "
          f"EV/1000 {r.ev_per_1000:6.1f} -> {r.ev_per_1000_corr:5.1f}   "
          f"EV {r.ev:8.0f} -> {r.ev_corretto:8.0f}")
tn = prov[prov.key=="TRENTO"].iloc[0]
print(f"\nQuota nazionale di Trento: {tn.ev/prov.ev.sum():.1%} -> {tn.ev_corretto/prov.ev_corretto.sum():.1%}")
print(f"Mediana EV/1000: {prov.ev_per_1000.median():.1f} -> {prov.ev_per_1000_corr.median():.1f}")


Totale nazionale: originale 1,370,592 -> corretto 1,370,592 (conservato)
Bacino flotte redistribuito: 175,144 EV (12.8% del totale)

Effetto sulle province più gonfiate (EV/1000 e totale EV: prima -> dopo):
  TRENTO                 f=88.9%  EV/1000  248.6 ->  31.9   EV    90849 ->    11644
  AOSTA                  f= 3.7%  EV/1000  189.6 -> 186.7   EV    17123 ->    16862
  FIRENZE                f=62.0%  EV/1000  120.7 ->  50.0   EV    82502 ->    34177
  BOLZANO                f=56.6%  EV/1000   76.3 ->  37.3   EV    26673 ->    13030
  REGGIO EMILIA          f=15.2%  EV/1000   59.1 ->  54.3   EV    21328 ->    19582
  TORINO                 f= 8.0%  EV/1000   52.2 ->  52.3   EV    83772 ->    83781
  ROMA                   f= 3.4%  EV/1000   51.8 ->  54.2   EV   148468 ->   155357
  COMO                   f= 0.6%  EV/1000   46.1 ->  50.0   EV    19317 ->    20953
  MILANO                 f= 1.1%  EV/1000   44.8 ->  48.4   EV    97285 ->   105220
  VARESE                 f= 0.1%  EV/

## 4. Ri-disaggregazione a livello di sezione

Il totale provinciale **corretto** viene ridistribuito alle sezioni con gli
**stessi pesi IDI×popolazione** già calcolati in produzione (`peso_EV`) e con lo
stesso cap di plausibilità (`cap_EV_sezione`, 1 EV per residente in età di
guida). Non ricalcoliamo l'IDI: cambia solo il totale da distribuire dentro
ciascuna provincia. Usiamo la stessa funzione di allocazione con cap e
redistribuzione dell'eccedenza della pipeline originale.

In [5]:
def alloca_con_cap(pesi, cap, totale, tol=1e-9):
    pesi = np.asarray(pesi, float); cap = np.asarray(cap, float); totale = float(totale)
    if totale <= tol: return np.zeros_like(pesi)
    if cap.sum() + tol < totale:
        raise ValueError(f"totale {totale:g} > capacita' {cap.sum():g}")
    alloc = np.zeros_like(pesi); res = totale; attive = (pesi>0)&(cap>0)
    while res > tol:
        capres = cap - alloc
        cand = attive & (capres > tol)
        if not cand.any(): raise RuntimeError("capacita' esaurita")
        pc = pesi[cand]
        if pc.sum() <= tol: pc = capres[cand]
        prop = res * pc / pc.sum()
        oltre = prop > capres[cand] + tol
        pos = np.flatnonzero(cand)
        if not oltre.any():
            alloc[pos] += prop; res = 0.0
        else:
            pcap = pos[oltre]; ass = capres[pcap]
            alloc[pcap] += ass; res -= ass.sum(); attive[pcap] = False
    return alloc

sez = pq.read_table(GAP_PARQUET,
    columns=["SEZ2011","PROVINCIA_EV_2025","peso_EV","cap_EV_sezione",
             "veicoli_da_ricaricare_stimati","flag_eleggibile_EV"]).to_pandas()
tot_corr = prov.set_index("PROVINCIA_EV_2025")["ev_corretto"]

sez["ev_corretto"] = 0.0
for p, idx in sez.groupby("PROVINCIA_EV_2025").groups.items():
    idx = list(idx)
    if p not in tot_corr.index:      # sezioni senza provincia riconosciuta (geometria senza statistiche)
        continue
    a = alloca_con_cap(sez.loc[idx,"peso_EV"].to_numpy(),
                       sez.loc[idx,"cap_EV_sezione"].to_numpy(), float(tot_corr[p]))
    sez.loc[idx,"ev_corretto"] = a

# verifiche
ric = sez.groupby("PROVINCIA_EV_2025").ev_corretto.sum()
scarto = (ric - tot_corr.reindex(ric.index)).abs().max()
print(f"Max scarto riconciliazione provinciale dopo ri-disaggregazione: {scarto:.2e}")
print(f"Totale nazionale sezioni corretto: {sez.ev_corretto.sum():,.0f} (atteso {prov.ev_corretto.sum():,.0f})")
print(f"Sezioni con domanda cambiata: {(np.abs(sez.ev_corretto - sez.veicoli_da_ricaricare_stimati)>1e-6).sum():,}")


Max scarto riconciliazione provinciale dopo ri-disaggregazione: 1.46e-11
Totale nazionale sezioni corretto: 1,370,592 (atteso 1,370,592)
Sezioni con domanda cambiata: 346,509


## 5. Salvataggio degli output

In [6]:
out_prov = prov[["PROVINCIA_EV_2025","ev","f","ev_privati","ev_flotta_redistr",
                 "ev_corretto","pop_guida","ev_per_1000","ev_per_1000_corr"]].copy()
out_prov = out_prov.rename(columns={"ev":"ev_originale","f":"quota_flotta_applicata"})
out_prov.to_csv(DIR/"domanda_ricarica_2025_per_provincia_CORRETTA.csv", index=False)

out_sez = sez[["SEZ2011","PROVINCIA_EV_2025","veicoli_da_ricaricare_stimati","ev_corretto"]].copy()
out_sez = out_sez.rename(columns={"veicoli_da_ricaricare_stimati":"ev_originale",
                                  "ev_corretto":"veicoli_da_ricaricare_corretto"})
out_sez.to_csv(DIR/"domanda_ricarica_2025_per_sezione_CORRETTA.csv", index=False)
print("Salvati:")
print("  domanda_ricarica_2025_per_provincia_CORRETTA.csv")
print("  domanda_ricarica_2025_per_sezione_CORRETTA.csv  (usato dal notebook gap v2)")


Salvati:
  domanda_ricarica_2025_per_provincia_CORRETTA.csv
  domanda_ricarica_2025_per_sezione_CORRETTA.csv  (usato dal notebook gap v2)


## Limiti dichiarati

1. **2019 vs 2025**: la quota-flotta è misurata sul parco 2019 (unico con la
   finalità d'uso). Il noleggio a lungo termine è cresciuto molto dopo il 2019,
   quindi le quote reali 2025 sono probabilmente **più alte**: la correzione è
   semmai **prudente** (sottostima l'entità delle flotte).
2. **Aosta** non è spiegata dalle flotte (quota 2019 ~4%): la sua inflazione
   deriva da immatricolazioni di concessionari/km0 registrate come `PROPRIO`.
   Per correggerla servirebbe un secondo criterio (es. cap di plausibilità sul
   tasso EV/1000). La versione qui non la tocca.
3. **Chiave di redistribuzione**: la popolazione in età di guida è una scelta
   neutra. Le auto aziendali circolano dove c'è attività economica, correlata
   ma non identica alla popolazione. La colonna `ev_corretto_alt` (redistribuzione
   proporzionale agli EV privati locali) è disponibile come analisi di
   sensibilità.
4. La correzione **non cambia** l'IDI né i pesi intra-provinciali: agisce solo
   sul totale provinciale, cioè esattamente sul punto in cui il bias entra.
5. **Interpretazione dei valori corretti**: dopo la correzione ogni provincia
   riflette la sua base di EV *residenti* più la quota di flotte che
   plausibilmente vi circola. Trentino resta sopra la mediana nazionale
   (~32 vs ~29 EV/1000): la correzione non "azzera" l'alta adozione locale, ma
   rimuove la parte contabile (auto targate lì ma circolanti altrove). Va letta
   come rimozione di un artefatto macroscopico, non come stima puntuale del
   parco residente.